In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load College Scorecard Field of Study data
scorecard_fos = pd.read_csv(r"..\data\raw\Most-Recent-Cohorts-Field-of-Study.csv")

# Load College Scorecard Institution Level data
scorecard_il = pd.read_csv(r"..\data\raw\Most-Recent-Cohorts-Institution.csv")

C:\Users\sigep\AppData\Local\Temp\ipykernel_10332\3539334973.py:5: DtypeWarning: Columns (9,1407,1408,1431,1432,1532,1537,1538,1539,1540,1542,1546,1589,1601,1602,1606,1608,1611,1614,1615,1616,1619,1620,1621,1622,1623,1624,1625,1626,1627,1628,1629,1653,1679,1690,1692,1697,1700,1702,1725,1726,1727,1728,1729,1743,1815,1816,1817,1818,1823,1824,1830,1831,1879,1880,1881,1882,1883,1884,1885,1886,1887,1888,1889,1890,1891,1892,1893,1894,1895,1896,1897,1898,1909,1910,1911,1912,1913,1957,1958,1959,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,1975,1976,1983,1984,2376,2377,2403,2404,2495,2496,2497,2498,2499,2500,2501,2502,2503,2504,2505,2506,2507,2508,2509,2510,2511,2512,2513,2514,2515,2516,2517,2518,2519,2520,2521,2522,2523,2524,2525,2526,2527,2528,2529,2530,2958,3215,3231,3235,3236) have mixed types. Specify dtype option on import or set low_memory=False.
  scorecard_il = pd.read_csv(r"..\data\raw\Most-Recent-Cohorts-Institution.csv")


In [3]:
fos_rename_map = {
    "UNITID" : "unitid",
    "INSTNM" : "inst_name",
    "CIPCODE" : "cipcode",
    "CIPDESC" : "cip_desc",
    "CREDLEV" : "credlev",
    "CREDDESC" : "cred_desc",
    "EARN_MDN_1YR" : "median_earnings_1yr",
    "EARN_NE_MDN_3YR" : "median_earnings_3yr",
    "EARN_MDN_5YR" : "median_earnings_5yr"
}

In [4]:
il_rename_map = {
    "UNITID" : "unitid",
    "STABBR" : "state",
    "CONTROL" : "control",
    "COSTT4_A" : "avg_annual_tuition_housing",
    "COSTT4_P" : "avg_annual_tuition",
    "NPT4_PUB" : "avg_net_price_pub",
    "NPT4_PRIV" : "avg_net_price_priv"
}

In [5]:
scorecard_fos_clean = scorecard_fos.rename(columns=fos_rename_map)
scorecard_il_clean  = scorecard_il.rename(columns=il_rename_map)

In [6]:
scorecard_fos_clean["unitid_n"] = (
    pd.to_numeric(scorecard_fos_clean["unitid"], errors="coerce")
      .astype("Int64")
      .astype(str)
)

scorecard_il_clean["unitid_n"] = (
    pd.to_numeric(scorecard_il_clean["unitid"], errors="coerce")
      .astype("Int64")
      .astype(str)
)

In [7]:
# Filter for institutions in Tennessee
scorecard_il_tn = scorecard_il_clean[scorecard_il_clean["state"] == "TN"]

tn_unitids = set(scorecard_il_tn["unitid_n"])

scorecard_fos_tn = scorecard_fos_clean[
    scorecard_fos_clean["unitid_n"].isin(tn_unitids)
]

In [8]:
# Join IL and FOS
scorecard_tn = scorecard_fos_tn.merge(
    scorecard_il_tn,
    on="unitid_n",
    how="left"
)

In [9]:
# Remove cip code 99 (placeholder for programs that cannot be included in traditional cip codes)
scorecard_tn = scorecard_tn[scorecard_tn["cipcode"] != "99"]

In [10]:
# Normalize cip4
scorecard_tn["cip4"] = (
    scorecard_tn["cipcode"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str[:4]
)

In [11]:
# NOrmalize credlev
scorecard_tn["credlev_n"] = (
    pd.to_numeric(scorecard_tn["credlev"], errors="coerce")
    .astype("Int64")
)

scorecard_tn["credlev_n"] = scorecard_tn["credlev_n"].astype(str)

In [12]:
# Normalize unitid 
scorecard_tn["unitid_n"] = scorecard_tn["unitid_n"].astype(int)

In [13]:
# Create cred_desc mapping
cred_desc_map = {1: "Certificate", 2: "Associate", 3: "Bachelor"}
scorecard_tn["cred_desc"] = scorecard_tn["credlev_n"].map(cred_desc_map)

In [14]:
# Create composite key
scorecard_tn["composite_key"] = (
    scorecard_tn["unitid_n"].astype(str) + "|" +
    scorecard_tn["cipcode"].astype(str) + "|" +
    scorecard_tn["credlev_n"].astype(str)
)

In [15]:
print("Rows:", len(scorecard_tn))
print("Unique composite keys:", scorecard_tn["composite_key"].nunique())
print("Duplicate composite keys:", 
      len(scorecard_tn) - scorecard_tn["composite_key"].nunique())

Rows: 4166
Unique composite keys: 4166
Duplicate composite keys: 0


In [16]:
# Export cleaned scorecard
scorecard_tn_export = scorecard_tn[[
    # IDs
    "unitid_n",
    
    # Institution info
    "inst_name",
    "state",
    "control",
    
    # Program info
    "cip4",
    "cip_desc",
    "credlev_n",
    "cred_desc",
    
    # Earnings
    "median_earnings_1yr",
    "median_earnings_3yr",
    "median_earnings_5yr",
    
    # Cost
    "avg_annual_tuition_housing",
    "avg_annual_tuition",
    "avg_net_price_pub",
    "avg_net_price_priv",
    
    # Join key
    "composite_key"
]]

In [17]:
scorecard_tn_export.head(2)

,unitid_n,inst_name,state,control,cip4,cip_desc,credlev_n,cred_desc,median_earnings_1yr,median_earnings_3yr,median_earnings_5yr,avg_annual_tuition_housing,avg_annual_tuition,avg_net_price_pub,avg_net_price_priv,composite_key
0,219505,American Baptist College,TN,2,2401,"Liberal Arts and Sciences, General Studies and...",2,NaN,PS,PS,PS,27241.0,NaN,NaN,19294.0,219505|2401|2
1,219505,American Baptist College,TN,2,2401,"Liberal Arts and Sciences, General Studies and...",3,NaN,PS,PS,PS,27241.0,NaN,NaN,19294.0,219505|2401|3


In [18]:
scorecard_tn_export.to_csv(
    "scorecard_tn_export.csv",
    index=False,
    encoding="utf-8"
)

In [19]:
# Load IPEDS 2024 Completions
completions_2024 = pd.read_csv(r"..\data\raw\c2024_a.csv")

In [20]:
# Keep only relevant columns
keep_cols = ["UNITID", "CIPCODE", "AWLEVEL", "CTOTALT", "CTOTALM", "CTOTALW"]
completions_2024 = completions_2024[keep_cols]

In [21]:
completions_2024.head(2)

,UNITID,CIPCODE,AWLEVEL,CTOTALT,CTOTALM,CTOTALW
0,100654,1.0999,5,12,2,10
1,100654,1.1001,5,10,1,9


In [22]:
# Remove cip code 99 (placeholder for programs that cannot be included in traditional cip codes)
completions_2024 = completions_2024[completions_2024["CIPCODE"] != "99"]

In [23]:
# Normalize column names
completions_2024 = completions_2024.rename(columns={
    "UNITID": "unitid",
    "CIPCODE": "cip6",
    "AWLEVEL": "awlevel",
    "CTOTALT": "completions",
    "CTOTALM": "completions_men",
    "CTOTALW": "completions_women"
})

In [24]:
# CIP6 is numeric xx.xxxx format; CIP4 = first 2+2 digits
completions_2024["cip4"] = completions_2024["cip6"].astype(str).str.replace(".", "", regex=False).str[:4]

In [25]:
credlev_map = {
    1: [2,3,5],   # certificate
    2: [4],       # associate
    3: [6]        # bachelor
}

def map_credlevel(awlevel):
    for proj_level, ipeds_codes in credlev_map.items():
        if awlevel in ipeds_codes:
            return proj_level
    return np.nan

completions_2024["credlev_proj"] = completions_2024["awlevel"].apply(map_credlevel)

In [26]:
# Drop rows outside 1,2,3
completions_2024 = completions_2024[completions_2024["credlev_proj"].notna()]
completions_2024["credlev_proj"] = completions_2024["credlev_proj"].astype(int)

In [27]:
# Add cred_desc
cred_desc_map = {1:"Certificate", 2:"Associate", 3:"Bachelor"}
completions_2024["cred_desc"] = completions_2024["credlev_proj"].map(cred_desc_map)

In [28]:
completions_2024["unitid_n"] = completions_2024["unitid"].astype(int)

In [29]:
print(completions_2024.columns.tolist())

['unitid', 'cip6', 'awlevel', 'completions', 'completions_men', 'completions_women', 'cip4', 'credlev_proj', 'cred_desc', 'unitid_n']


In [30]:
completions_2024_agg = (
    completions_2024
    .groupby(["unitid_n","cip4","credlev_proj","cred_desc"], as_index=False)
    .agg({
        "completions":"sum",
        "completions_men":"sum",
        "completions_women":"sum"
    })
)

In [31]:
# Create composite key
completions_2024_agg["composite_key"] = (
    completions_2024_agg["unitid_n"].astype(str) + "|" +
    completions_2024_agg["cip4"].astype(str) + "|" +
    completions_2024_agg["credlev_proj"].astype(str)
)

In [33]:
completions_2024_agg.to_csv("completions_2024_agg.csv", index=False, encoding="utf-8")